In [1]:
import torch
from torch import nn
from torch.nn import functional as F
import numpy as np

In [2]:
class RNNLayer(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.Wxh = nn.Parameter(torch.randn(input_size, hidden_size) * 0.01)
        self.Whh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
        self.Bh = nn.Parameter(torch.zeros(hidden_size))

        self.Why = nn.Parameter(torch.randn(hidden_size, output_size) * 0.01)
        self.By = nn.Parameter(torch.zeros(output_size))

    def forward(self, x, h_prev):
        h_next = torch.tanh(x @ self.Wxh + h_prev @ self.Whh + self.Bh) # [batch_size, hidden_size]
        y = h_next @ self.Why + self.By                                 # [batch_size, output_size]

        return h_next, y

In [3]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, output_size, pad_token_id, lr=0.001):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn_layer = RNNLayer(embedding_dim, hidden_size, output_size)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=pad_token_id)
        self.optimizer = torch.optim.Adam(self.parameters(), lr=lr)
        self.pad_token_id = pad_token_id

    def forward(self, x):
        batch_size, seq_len = x.size()
        h_prev = torch.zeros(batch_size, self.rnn_layer.Whh.size(0)).to(x.device) # [batch_size, hidden_size]

        x_mask = (x == self.pad_token_id).unsqueeze(-1)                           # [batch_size, seq_len, 1]
        x_emb = self.embedding(x)                                                 # [batch_size, seq_len, embedding_dim]
        x_emb = x_emb.masked_fill(x_mask, 0)                                      

        outputs = []
        for t in range(seq_len):
            x_t = x_emb[:, t, :]                                                  # [batch_size, embedding_dim]
            h_prev, y_t = self.rnn_layer(x_t, h_prev)                             # [batch_size, hidden_size], [batch_size, output_size]
            outputs.append(y_t)
            
        
        return torch.stack(outputs, dim=1)                                        # [batch_size, seq_len, output_size]
    
    def backward(self, y_pred, y_true):
        loss = self.loss_fn(y_pred.view(-1, y_pred.size(-1)), y_true.view(-1))
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return loss.item()
    
    def fit(self, x, y, epochs=10, print_every=1):
        for epoch in range(epochs):
            y_pred = self.forward(x)
            loss = self.backward(y_pred, y)
            if (epoch + 1) % print_every == 0:
                print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}")

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

src_texts = [
    "I love machine learning.",
    "Transformers are powerful models.",
    "I learn deep learning."
]

src_tokens = tokenizer(src_texts, padding=True, truncation=True, return_tensors="pt")

print("Source tokens:\n", src_tokens['input_ids'])
print("Source tokens shape:", src_tokens['input_ids'].shape)

Source tokens:
 tensor([[   40,  1842,  4572,  4673,    13, 50256],
        [41762,   364,   389,  3665,  4981,    13],
        [   40,  2193,  2769,  4673,    13, 50256]])
Source tokens shape: torch.Size([3, 6])


In [5]:
rnn = SimpleRNN(
    vocab_size=tokenizer.vocab_size,
    embedding_dim=64,
    hidden_size=128,
    output_size=tokenizer.vocab_size,
    pad_token_id=tokenizer.pad_token_id,
    lr=0.001
)
rnn.fit(src_tokens['input_ids'], src_tokens['input_ids'], epochs=100, print_every=10)

Epoch 10/100, Loss: 10.3306
Epoch 20/100, Loss: 8.4482
Epoch 30/100, Loss: 5.4323
Epoch 40/100, Loss: 3.0056
Epoch 50/100, Loss: 1.9912
Epoch 60/100, Loss: 1.5474
Epoch 70/100, Loss: 1.2594
Epoch 80/100, Loss: 1.0092
Epoch 90/100, Loss: 0.8038
Epoch 100/100, Loss: 0.6340


In [6]:
def generate(model, tokenizer, prompt, max_length=20):
    model.eval()
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    generated = input_ids

    with torch.no_grad():
        for _ in range(max_length):
            outputs = model(generated)
            next_token_logits = outputs[:, -1, :]
            next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)
            generated = torch.cat((generated, next_token_id), dim=1)

    return tokenizer.decode(generated[0], skip_special_tokens=True)

In [7]:
for text in src_texts:
    print(f"Prompt: {text}")
    print("Generated:", generate(rnn, tokenizer, text))
    print()

Prompt: I love machine learning.
Generated: I love machine learning.....................

Prompt: Transformers are powerful models.
Generated: Transformers are powerful models.....................

Prompt: I learn deep learning.
Generated: I learn deep learning.....................

